In [1]:
import os
import pandas as pd
from collections import Counter, defaultdict




In [2]:
DATA_ROOT = "/kaggle/input/h-and-m-personalized-fashion-recommendations"
transaction_path = os.path.join(DATA_ROOT, "transactions_train.csv")
chunksize = 10_000_000

# Use a tighter recent window (10 days) to capture the most recent buying signals
RECENT_DAYS = 10

max_date = None
for chunk in pd.read_csv(
    transaction_path,
    usecols=["t_dat"],
    dtype=str,
    chunksize=chunksize,
):
    dates = pd.to_datetime(chunk["t_dat"], format="%Y-%m-%d", errors="coerce")
    chunk_max = dates.max()
    if pd.isna(chunk_max):
        continue
    if max_date is None or chunk_max > max_date:
        max_date = chunk_max

if max_date is None:
    raise RuntimeError("Unable to determine max transaction date.")

recent_threshold = max_date - pd.Timedelta(days=RECENT_DAYS)




In [3]:
article_counter = Counter()
cust_counter = defaultdict(Counter)  # per‑customer frequency of recent articles

for chunk in pd.read_csv(
    transaction_path,
    usecols=["t_dat", "customer_id", "article_id"],
    dtype=str,
    chunksize=chunksize,
):
    chunk["t_dat"] = pd.to_datetime(chunk["t_dat"], format="%Y-%m-%d", errors="coerce")
    recent_mask = chunk["t_dat"] >= recent_threshold
    recent_chunk = chunk[recent_mask]

    article_counter.update(recent_chunk["article_id"])

    for cust_id, art_id in zip(recent_chunk["customer_id"], recent_chunk["article_id"]):
        cust_counter[cust_id][art_id] += 1

top_12_articles = [article for article, _ in article_counter.most_common(12)]

cust_top12_list = {
    cust_id: [art for art, _ in counter.most_common(12)]
    for cust_id, counter in cust_counter.items()
}




In [4]:
sample_sub_path = os.path.join(DATA_ROOT, "sample_submission.csv")
submission_df = pd.read_csv(sample_sub_path, dtype=str)




In [5]:
def build_prediction(cid: str) -> str:
    """Return a space‑separated string of up to 12 article IDs for a customer."""
    personal = cust_top12_list.get(cid, []).copy()
    if len(personal) < 12:
        for art in top_12_articles:
            if art not in personal:
                personal.append(art)
            if len(personal) == 12:
                break
    return " ".join(personal)


submission_df["prediction"] = submission_df["customer_id"].apply(build_prediction)




In [6]:
output_path = "submission.csv"
submission_df.to_csv(output_path, index=False)
print(f"Submission written to {output_path}. Shape: {submission_df.shape}")

Submission written to submission.csv. Shape: (1371980, 2)
